# 🕷️ Парсер фейків — Укрінформ + StopFake

| Сайт | Метод | ~Статей |
|------|-------|---------|
| ukrinform.ua/tag-fejk | infinite scroll → парсимо статті напряму | ~30 |
| stopfake.org/uk/category/factcheck_facebook_ua/ | сторінки /page/N/ | ~4500 |

**Мітка: 1 = FAKE**

In [ ]:
!pip install requests beautifulsoup4 lxml tqdm -q

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, re, os
from tqdm import tqdm

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36',
    'Accept-Language': 'uk-UA,uk;q=0.9',
    'Accept': 'text/html,application/xhtml+xml',
    'Referer': 'https://www.google.com/',
}
FAKE_LABEL = 1
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def get_soup(url, pause=1.2):
    time.sleep(pause)
    r = SESSION.get(url, timeout=15)
    r.raise_for_status()
    r.encoding = 'utf-8'
    return BeautifulSoup(r.text, 'lxml'), r.status_code

def clean(text):
    return re.sub(r'\s+', ' ', text).strip().strip('.,; ')

print('✅ Готово')

✅ Готово


---
## 📰 ЧАСТИНА 1 — Укрінформ

Сайт використовує **infinite scroll** — пагінації немає.
Стратегія: збираємо посилання на статті з першої сторінки → заходимо у кожну → беремо текст.

Посилання статей мають вигляд: `/rubric-war/XXXXXXX-nazva.html`

In [6]:
UKRINFORM_BASE = 'https://www.ukrinform.ua'
UKRINFORM_TAG  = 'https://www.ukrinform.ua/tag-fejk'

soup_uk, code = get_soup(UKRINFORM_TAG)
print(f'Status: {code}')

# Збираємо ТІЛЬКИ посилання на статті (формат /rubric-XXXX/NNNNNN-title.html)
article_links = []
for a in soup_uk.find_all('a', href=True):
    href = a['href']
    # Статті Укрінформу: /rubric-*/ЦИФРИ-назва.html
    if re.match(r'^/rubric-[a-z]+/\d+-.+\.html$', href):
        full = UKRINFORM_BASE + href
        if full not in article_links:
            article_links.append(full)

print(f'Знайдено посилань на статті: {len(article_links)}')
for l in article_links[:5]:
    print(f'  {l}')

Status: 200
Знайдено посилань на статті: 45
  https://www.ukrinform.ua/rubric-world/4101357-situacia-na-blizkomu-shodi-pokazue-so-konflikt-stav-globalnim-ceski-eksperti.html
  https://www.ukrinform.ua/rubric-society/4101356-zadusili-j-vikinuli-z-vikna-v-italii-pidtverdili-vbivstvo-ukrainskogo-bankira-adarica.html
  https://www.ukrinform.ua/rubric-regions/4101355-rosiani-kabami-atakuvali-sumsinu-troe-poranenih.html
  https://www.ukrinform.ua/rubric-world/4101353-iran-zavdav-raketnogo-udaru-po-centralnij-castini-izrailu.html
  https://www.ukrinform.ua/rubric-regions/4101347-punkt-propusku-tisa-na-kordoni-z-ugorsinou-prizupiniv-robotu-nemae-svitla.html


In [3]:
def parse_ukrinform_article(url):
    """Парсить заголовок + лід зі статті Укрінформу."""
    try:
        soup, _ = get_soup(url, pause=1.0)
        
        h1 = soup.find('h1')
        title = clean(h1.get_text()) if h1 else ''
        
        # Лід (короткий опис під заголовком)
        lead = ''
        for sel in ['.articleLead', '.article__lead', '.lead', '.article-lead', 'p.lead']:
            el = soup.select_one(sel)
            if el:
                lead = clean(el.get_text())
                break
        
        # Якщо ліду нема — перший абзац тексту
        if not lead:
            for sel in ['.articleText', '.article__text', '.article-text']:
                block = soup.select_one(sel)
                if block:
                    paras = [clean(p.get_text()) for p in block.find_all('p') if len(p.get_text(strip=True)) > 40]
                    if paras:
                        lead = paras[0][:250]
                    break
        
        text = f'{title}. {lead}'.strip('. ') if lead else title
        text = clean(text)[:400]
        return text if len(text) > 20 else None
    except:
        return None


uk_results = []
print(f'🕷️ Парсимо {len(article_links)} статей Укрінформу...')

for url in tqdm(article_links, desc='Укрінформ'):
    text = parse_ukrinform_article(url)
    if text:
        uk_results.append({'text': text, 'label': FAKE_LABEL, 'source': 'ukrinform'})

df_ukrinform = pd.DataFrame(uk_results).drop_duplicates(subset='text')
print(f'\n✅ Укрінформ: {len(df_ukrinform)} унікальних статей')
print('\nПриклади:')
for _, r in df_ukrinform.head(3).iterrows():
    print(f'  {r["text"][:100]}')

🕷️ Парсимо 45 статей Укрінформу...


Укрінформ: 100%|██████████| 45/45 [01:01<00:00,  1.37s/it]


✅ Укрінформ: 45 унікальних статей

Приклади:
  Ситуація на Близькому Сході показує, що конфлікт став глобальним - чеські експерти
  Задушили й викинули з вікна: в Італії підтвердили вбивство українського банкіра Адаріча
  Росіяни КАБами атакували Сумщину - троє поранених


---
## 📰 ЧАСТИНА 2 — StopFake

URL: https://www.stopfake.org/uk/category/factcheck_facebook_ua/

WordPress тема Newspaper — селектор `div.td_module_10` (10 постів на сторінці, 382 сторінки = ~3800 статей).

**Запускай по порядку: `stopfake_parse_fn` → `stopfake_pagination_test` → `stopfake_scrape`**

In [11]:

SF_SELECTOR = 'div.td_module_10'
STOPFAKE_BASE = 'https://www.stopfake.org'

STOPFAKE_URL = 'https://www.stopfake.org/uk/category/factcheck_facebook_ua/'

def get_sf_page_url(n):
    if n == 1:
        return STOPFAKE_URL
    return STOPFAKE_URL.rstrip('/') + f'/page/{n}/'

def parse_stopfake_item(item):
    """
    Парсить заголовок статті (= сам фейк).
    НЕ беремо excerpt — там спростування.
    """
    title_tag = item.select_one('.entry-title, h3, h4, h2')
    if not title_tag:
        return None, ''

    a = title_tag.find('a')
    title = clean(a.get_text() if a else title_tag.get_text())

    # Прибираємо префікс "Фейк:" — він є маркером але не частина самого фейку
    title = re.sub(r'^(Фейк|Fake|ФЕЙК|Фактчек|Маніпуляція)\s*[:\-]\s*', '', title, flags=re.IGNORECASE).strip()

    href = ''
    a_tag = item.find('a', href=True)
    if a_tag:
        href = a_tag['href']
        if href.startswith('/'):
            href = STOPFAKE_BASE + href

    return clean(title), href


# Тест
soup_sf, _ = get_soup(STOPFAKE_URL, pause=1.0)
items_test = soup_sf.select(SF_SELECTOR)
print(f'Знайдено елементів: {len(items_test)}')
print()
print('📋 Тест — тільки заголовки:')
for item in items_test[:5]:
    text, href = parse_stopfake_item(item)
    if text:
        print(f'  ❌ {text}')


Знайдено елементів: 10

📋 Тест — тільки заголовки:
  ❌ В Дубаї за мародерство після обстрілів затримали 19 українців — Euronews
  ❌ Через новий закон про евакуацію дітей по всій Україні будуть «силою відбирати та
  ❌ Іран атакує США та Ізраїль ракетами, переданими Україні — USA Today
  ❌ В Україні хочуть скасувати 8 березня і «залишити жінок без квітів»
  ❌ У Дубаї під час атаки повністю згорів «маєток одного з колишніх заступників Сирського»


In [12]:
for test_page in [2, 3]:
    url = get_sf_page_url(test_page)
    try:
        soup_t, code_t = get_soup(url, pause=1.0)
        items_t = soup_t.select(SF_SELECTOR)
        print(f'Сторінка {test_page}: {code_t} | {len(items_t)} елементів')
        if items_t:
            t, _ = parse_stopfake_item(items_t[0])
            print(f'  Перший: {t[:80]}')
    except Exception as e:
        print(f'Сторінка {test_page} помилка: {e}')


Сторінка 2: 200 | 10 елементів
  Перший: Україна «скасувала виплати» сім’ям зниклих безвісти військових
Сторінка 3: 200 | 10 елементів
  Перший: Українське зерно викликало у Франції спалах дерматиту в худоби


In [13]:
MAX_PAGES_SF  = 382 
PAUSE_SF      = 1.5 
PROGRESS_FILE = 'stopfake_progress_deffake.csv'

# Завантажуємо прогрес якщо є
if os.path.exists(PROGRESS_FILE):
    df_prog = pd.read_csv(PROGRESS_FILE)
    sf_results = df_prog.to_dict('records')
    done_pages = int(df_prog['page'].max())
    seen_texts = set(df_prog['text'].tolist())
    print(f'Продовжуємо з сторінки {done_pages+1} ({len(sf_results)} статей вже є)')
else:
    sf_results = []
    done_pages = 0
    seen_texts = set()

print(f'🕷️  Парсимо StopFake: сторінки {done_pages+1}–{MAX_PAGES_SF}')
print(f'    Очікувано ще: ~{(MAX_PAGES_SF - done_pages) * 12} статей')
print()

for page in tqdm(range(done_pages + 1, MAX_PAGES_SF + 1), desc='StopFake'):
    url = get_sf_page_url(page)
    try:
        soup_page, code = get_soup(url, pause=PAUSE_SF)
        
        if code == 404:
            print(f'\nСторінка {page}: 404 — досягнуто кінця списку')
            break
        
        items_page = soup_page.select(SF_SELECTOR) if SF_SELECTOR else []
        
        if not items_page:
            print(f'\nСторінка {page}: статей не знайдено — можливо змінився селектор')
            # Спробуємо перевизначити селектор
            SF_SELECTOR_NEW, items_page = find_best_selector(soup_page)
            if SF_SELECTOR_NEW and items_page:
                print(f'   Новий селектор: {SF_SELECTOR_NEW}')
                SF_SELECTOR = SF_SELECTOR_NEW
            else:
                continue
        
        added = 0
        for item in items_page:
            text, _ = parse_stopfake_item(item)
            if text and len(text) > 20 and text not in seen_texts:
                seen_texts.add(text)
                sf_results.append({'text': text, 'label': FAKE_LABEL,
                                   'source': 'stopfake', 'page': page})
                added += 1
    
    except requests.exceptions.HTTPError as e:
        if e.response is not None and e.response.status_code == 404:
            print(f'\nКінець на сторінці {page}')
            break
        time.sleep(5)
        continue
    except Exception as e:
        print(f'\nПомилка сторінки {page}: {str(e)[:80]}')
        time.sleep(3)
        continue
    
    if page % 10 == 0:
        pd.DataFrame(sf_results).to_csv(PROGRESS_FILE, index=False, encoding='utf-8')

df_stopfake = pd.DataFrame(sf_results).drop_duplicates(subset='text')
df_stopfake.to_csv(PROGRESS_FILE, index=False, encoding='utf-8')

print(f'\nStopFake: {len(df_stopfake)} унікальних статей')
print('\nПриклади:')
for _, r in df_stopfake.sample(min(3, len(df_stopfake))).iterrows():
    print(f'  {r["text"][:100]}')

🕷️  Парсимо StopFake: сторінки 1–382
    Очікувано ще: ~4584 статей



StopFake:   0%|          | 0/382 [00:00<?, ?it/s]

StopFake: 100%|██████████| 382/382 [10:36<00:00,  1.67s/it]


StopFake: 3812 унікальних статей

Приклади:
  священнослужителів УПЦ МП, які прийшли до Зеленського, вигнали «хибною» повітряною тривогою
  Напередодні контрнаступу у Британії вербували найманців для України
  Українські націоналісти захопили у Краматорську 20 автомобілів СММ ОБСЄ


In [14]:

SF_SELECTOR = 'div.td_module_10'
STOPFAKE_BASE = 'https://www.stopfake.org'

STOPFAKE_URL = 'https://www.stopfake.org/uk/category/covid19_ua/'

def get_sf_page_url(n):
    if n == 1:
        return STOPFAKE_URL
    return STOPFAKE_URL.rstrip('/') + f'/page/{n}/'

def parse_stopfake_item(item):
    title_tag = item.select_one('.entry-title, h3, h4, h2')
    if not title_tag:
        return None, ''

    a = title_tag.find('a')
    title = clean(a.get_text() if a else title_tag.get_text())

    title = re.sub(r'^(Фейк|Fake|ФЕЙК|Фактчек|Маніпуляція)\s*[:\-]\s*', '', title, flags=re.IGNORECASE).strip()

    href = ''
    a_tag = item.find('a', href=True)
    if a_tag:
        href = a_tag['href']
        if href.startswith('/'):
            href = STOPFAKE_BASE + href

    return clean(title), href


soup_sf, _ = get_soup(STOPFAKE_URL, pause=1.0)
items_test = soup_sf.select(SF_SELECTOR)
print(f'Знайдено елементів: {len(items_test)}')
print()
print('📋 Тест — тільки заголовки:')
for item in items_test[:5]:
    text, href = parse_stopfake_item(item)
    if text:
        print(f'  ❌ {text}')


Знайдено елементів: 10

📋 Тест — тільки заголовки:
  ❌ Андріївка – явка з повинною окупантів: дайджест пропаганди РФ за 15 серпня
  ❌ У біолабораторіях України проводили досліди з коронавірусом кажанів
  ❌ Білл Гейтс закликав негайно прибрати з ринку усі вакцини COVID-19
  ❌ Королеву Єлизавету II лікують від COVID-19 «забороненими» ліками
  ❌ Вакцини проти COVID-19 збільшили дитячу смертність у 52 рази


In [15]:
for test_page in [2, 3]:
    url = get_sf_page_url(test_page)
    try:
        soup_t, code_t = get_soup(url, pause=1.0)
        items_t = soup_t.select(SF_SELECTOR)
        print(f'Сторінка {test_page}: {code_t} | {len(items_t)} елементів')
        if items_t:
            t, _ = parse_stopfake_item(items_t[0])
            print(f'  Перший: {t[:80]}')
    except Exception as e:
        print(f'Сторінка {test_page} помилка: {e}')


Сторінка 2: 200 | 10 елементів
  Перший: Відеофейк: Смерть дитини після щеплення
Сторінка 3: 200 | 10 елементів
  Перший: Міжнародний суд у Гаазі визнав пандемію COVID-19 геноцидом і фейком


In [17]:
MAX_PAGES_SF  = 79
PAUSE_SF      = 1.5 
PROGRESS_FILE = 'stopfake_progress_covidfake.csv'

# Завантажуємо прогрес якщо є
if os.path.exists(PROGRESS_FILE):
    df_prog = pd.read_csv(PROGRESS_FILE)
    sf_results = df_prog.to_dict('records')
    done_pages = int(df_prog['page'].max())
    seen_texts = set(df_prog['text'].tolist())
    print(f'Продовжуємо з сторінки {done_pages+1} ({len(sf_results)} статей вже є)')
else:
    sf_results = []
    done_pages = 0
    seen_texts = set()

print(f'    Парсимо StopFake: сторінки {done_pages+1}–{MAX_PAGES_SF}')
print(f'    Очікувано ще: ~{(MAX_PAGES_SF - done_pages) * 12} статей')
print()

for page in tqdm(range(done_pages + 1, MAX_PAGES_SF + 1), desc='StopFake'):
    url = get_sf_page_url(page)
    try:
        soup_page, code = get_soup(url, pause=PAUSE_SF)
        
        if code == 404:
            print(f'\nСторінка {page}: 404 — досягнуто кінця списку')
            break
        
        items_page = soup_page.select(SF_SELECTOR) if SF_SELECTOR else []
        
        if not items_page:
            print(f'\nСторінка {page}: статей не знайдено — можливо змінився селектор')
            # Спробуємо перевизначити селектор
            SF_SELECTOR_NEW, items_page = find_best_selector(soup_page)
            if SF_SELECTOR_NEW and items_page:
                print(f'   Новий селектор: {SF_SELECTOR_NEW}')
                SF_SELECTOR = SF_SELECTOR_NEW
            else:
                continue
        
        added = 0
        for item in items_page:
            text, _ = parse_stopfake_item(item)
            if text and len(text) > 20 and text not in seen_texts:
                seen_texts.add(text)
                sf_results.append({'text': text, 'label': FAKE_LABEL,
                                   'source': 'stopfake', 'page': page})
                added += 1
    
    except requests.exceptions.HTTPError as e:
        if e.response is not None and e.response.status_code == 404:
            print(f'\nКінець на сторінці {page}')
            break
        time.sleep(5)
        continue
    except Exception as e:
        print(f'\nПомилка сторінки {page}: {str(e)[:80]}')
        time.sleep(3)
        continue
    
    if page % 10 == 0:
        pd.DataFrame(sf_results).to_csv(PROGRESS_FILE, index=False, encoding='utf-8')

df_stopfake = pd.DataFrame(sf_results).drop_duplicates(subset='text')
df_stopfake.to_csv(PROGRESS_FILE, index=False, encoding='utf-8')

print(f'\nStopFake: {len(df_stopfake)} унікальних статей')
print('\nПриклади:')
for _, r in df_stopfake.sample(min(3, len(df_stopfake))).iterrows():
    print(f'  {r["text"][:100]}')

    Парсимо StopFake: сторінки 1–79
    Очікувано ще: ~948 статей



StopFake: 100%|██████████| 79/79 [02:41<00:00,  2.04s/it]


StopFake: 779 унікальних статей

Приклади:
  Батьки з Чернігова відстояли право вчителів ходити на роботу «без уколів»
  Київське метро відновить свою роботу 31 травня
  Фотофейк: Перша жертва вакцинації в Чернігові
